# 路径管理

In [2]:
import sys

sdk_path = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/FDCL_v2/SDK"

if sdk_path not in sys.path:
    sys.path.append(sdk_path)

## 数据预处理

In [5]:
import os
import yaml
import numpy as np
import pandas as pd
import torch

# =========================================================
# 配置文件路径
# =========================================================
config_path = "/home/charles/HZU/Industrial_Software_Testing/Industrial_Software_Testing/FDCL_v2/CPC/config.yaml"

with open(config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# 读取 stage1 数据路径
stage1_cfg = cfg["stage1"]
data_paths = stage1_cfg["data_path"]

train_path = data_paths["train_path"]
val_path   = data_paths["val_path"]
test_path  = data_paths["test_path"]

# 检查文件是否存在
for path, name in [(train_path, "train"), (val_path, "val"), (test_path, "test")]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"找不到 stage1 {name} 文件: {path}")

# =========================================================
# 指定要读取的列
# =========================================================
feature_cols = ["vibration_ch1", "vibration_ch2", "vibration_ch3", "vibration_ch4"]
label_col = "class_id"

# =========================================================
# 只读取需要的列
# =========================================================
use_cols = feature_cols + [label_col]

train_df = pd.read_csv(train_path, usecols=use_cols)
val_df   = pd.read_csv(val_path, usecols=use_cols)
test_df  = pd.read_csv(test_path, usecols=use_cols)

print("Stage1 train shape:", train_df.shape)
print("Stage1 val shape  :", val_df.shape)
print("Stage1 test shape :", test_df.shape)

# =========================================================
# 提取特征和标签
# =========================================================
def extract_features_labels(df, feature_cols, label_col):
    X = df[feature_cols].values.astype(np.float32)
    y = df[label_col].values.astype(np.int64).reshape(-1)
    return X, y

X_train, y_train = extract_features_labels(train_df, feature_cols, label_col)
X_val, y_val     = extract_features_labels(val_df, feature_cols, label_col)
X_test, y_test   = extract_features_labels(test_df, feature_cols, label_col)

print("\nRaw feature shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

print("\nLabel statistics:")
print("Train unique classes:", np.unique(y_train))
print("Val   unique classes:", np.unique(y_val))
print("Test  unique classes:", np.unique(y_test))

# =========================================================
# 转换为 PyTorch Tensor
# =========================================================
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_val   = torch.tensor(y_val, dtype=torch.long)
y_test  = torch.tensor(y_test, dtype=torch.long)

print("\nFinal tensor shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)

Stage1 train shape: (15000, 5)
Stage1 val shape  : (5000, 5)
Stage1 test shape : (5000, 5)

Raw feature shapes:
X_train: (15000, 4) y_train: (15000,)
X_val  : (5000, 4) y_val  : (5000,)
X_test : (5000, 4) y_test : (5000,)

Label statistics:
Train unique classes: [0 1 3 4 5]
Val   unique classes: [0 1 3 4 5]
Test  unique classes: [0 1 3 4 5]

Final tensor shapes:
X_train: torch.Size([15000, 4]) y_train: torch.Size([15000])
X_val  : torch.Size([5000, 4]) y_val  : torch.Size([5000])
X_test : torch.Size([5000, 4]) y_test : torch.Size([5000])


## CPC动态特征计算

### CPC模型定义

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CPCEncoder(nn.Module):
    """编码器：将原始输入映射到潜在表示 z_t"""
    def __init__(self, input_dim, hidden_dim, z_dim):
        super().__init__()
        # 示例：一个简单的卷积编码器（适用于时序数据）
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, z_dim, kernel_size=3, stride=1, padding=1)
        )
    
    def forward(self, x):
        # x 形状: (batch, input_dim, seq_len)
        return self.conv(x)  # 输出: (batch, z_dim, seq_len)

class CPCAutoregressive(nn.Module):
    """自回归模型：根据过去的 z 生成上下文向量 c_t"""
    def __init__(self, z_dim, hidden_dim, context_dim):
        super().__init__()
        self.gru = nn.GRU(z_dim, hidden_dim, num_layers=2, batch_first=True, bidirectional=False)
        self.fc = nn.Linear(hidden_dim, context_dim)
    
    def forward(self, z):
        # z 形状: (batch, seq_len, z_dim)
        gru_out, _ = self.gru(z)  # gru_out: (batch, seq_len, hidden_dim)
        context = self.fc(gru_out)  # (batch, seq_len, context_dim)
        return context

class CPCPredictor(nn.Module):
    """预测头：对于每一个未来步 k，将上下文向量映射到预测表示"""
    def __init__(self, context_dim, z_dim, num_steps=5):
        super().__init__()
        self.num_steps = num_steps
        # 为每一个预测步 k 学习一个线性变换矩阵 W_k
        self.Wk = nn.ModuleList([
            nn.Linear(context_dim, z_dim, bias=False) for _ in range(num_steps)
        ])
    
    def forward(self, context, k):
        # context: (batch, seq_len, context_dim)
        # 返回对第 k 步未来的预测表示
        return self.Wk[k-1](context)  # (batch, seq_len, z_dim)

class CPCModel(nn.Module):
    """CPC 整体模型"""
    def __init__(self, input_dim, encoder_hidden_dim=256, z_dim=128,
                 ar_hidden_dim=256, context_dim=128, pred_steps=5):
        super().__init__()
        self.pred_steps = pred_steps
        self.z_dim = z_dim
        
        self.encoder = CPCEncoder(input_dim, encoder_hidden_dim, z_dim)
        self.ar = CPCAutoregressive(z_dim, ar_hidden_dim, context_dim)
        self.predictor = CPCPredictor(context_dim, z_dim, pred_steps)
    
    def forward(self, x):
        # x: (batch, input_dim, seq_len)
        batch_size, _, seq_len = x.shape
        
        # 1. 编码得到 z
        z = self.encoder(x)                     # (batch, z_dim, seq_len)
        z = z.permute(0, 2, 1)                  # (batch, seq_len, z_dim)
        
        # 2. 生成上下文向量 c_t
        context = self.ar(z)                    # (batch, seq_len, context_dim)
        
        # 3. 收集正负样本并计算 InfoNCE 损失
        loss, accuracy = self._compute_infonce(z, context)
        return loss, accuracy
    
    def _compute_infonce(self, z, context):
        """
        计算 InfoNCE 损失。
        对每个时间步 t 和每个预测步 k，真实未来 z_{t+k} 为正样本，
        同一批次内其他时间步（或随机采样）的 z 为负样本。
        """
        batch_size, seq_len, z_dim = z.shape
        total_loss = 0.0
        correct = 0
        total = 0
        
        # 循环每个预测步 k
        for k in range(1, self.pred_steps + 1):
            # 预测对第 k 步未来的表示: W_k c_t
            pred = self.predictor(context, k)           # (batch, seq_len, z_dim)
            
            # 正样本：真实的未来 z_{t+k}
            # 为了对齐时间维度，截取有效部分
            t_max = seq_len - k
            if t_max <= 0:
                continue
            
            # 取对应时间的预测和真实未来
            pred_valid = pred[:, :t_max, :]              # (batch, t_max, z_dim)
            z_future = z[:, k:, :]                       # (batch, t_max, z_dim)
            
            # 计算相似度分数（点积）
            # pred_valid: (batch, t_max, z_dim) -> 展平为 (batch * t_max, z_dim)
            # z_all: (batch * seq_len, z_dim) 作为所有候选负样本
            z_all_flat = z.reshape(-1, z_dim)            # (batch * seq_len, z_dim)
            
            # 预测与所有候选的相似度矩阵
            pred_flat = pred_valid.reshape(-1, z_dim)    # (batch * t_max, z_dim)
            logits = torch.matmul(pred_flat, z_all_flat.T)  # (batch * t_max, batch * seq_len)
            
            # 构造标签：正样本的位置索引
            # 对于每个 (batch_i, t)，正样本是同一序列同一批次的未来时间步
            indices = torch.arange(batch_size * t_max, device=z.device)
            labels = (indices // t_max) * seq_len + (indices % t_max) + k
            labels = labels.long()
            
            # 交叉熵损失
            loss_k = F.cross_entropy(logits, labels)
            total_loss += loss_k
            
            # 计算准确率
            pred_idx = logits.argmax(dim=1)
            correct += (pred_idx == labels).sum().item()
            total += labels.size(0)
        
        avg_loss = total_loss / self.pred_steps
        accuracy = correct / total if total > 0 else 0.0
        return avg_loss, accuracy

    def get_representations(self, x):
        """提取特征表示（上下文向量或潜在向量），供下游任务使用"""
        z = self.encoder(x).permute(0, 2, 1)  # (batch, seq_len, z_dim)
        context = self.ar(z)                  # (batch, seq_len, context_dim)
        return context   # 也可选择返回 z
